# Lekcia 07 - Vzor plánovania návrhu

Tento zápisník demonštruje **vzor plánovania návrhu** pre AI agentov pomocou Microsoft Agent Framework.
Naučíte sa, ako rozdeliť zložité cestovné požiadavky na štruktúrované podúlohy, prideliť ich špecializovaným agentom,
a vykonať výsledný plán — všetko pomocou štruktúrovaného výstupu podporeného modelmi Pydantic.


## Nastavenie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Rozklad úlohy

Rozklad úlohy je jadrom návrhového vzoru plánovania. Namiesto toho, aby sme požiadali jediného agenta,
aby zvládol zložitú požiadavku od začiatku do konca, rozdelíme problém na menšie, dobre definované **podúlohy**.
Každej podúlohe je pridelený špecializovaný agent (napr. lety, hotely, aktivity) s jasnými
prioritami a poradím závislostí.

Tento prístup prináša niekoľko výhod:
- **Jasnosť**: každá podúloha má jednoznačnú zodpovednosť.
- **Paralelizmus**: nezávislé podúlohy môžu bežať súbežne.
- **Spoľahlivosť**: zlyhania sú izolované na jednotlivé podúlohy.
- **Sledovanie rozpočtu**: náklady sa odhadujú pre každú podúlohu a sčítavajú sa.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Vytvorenie plniaceho agenta s štruktúrovaným výstupom

Plniaci agent funguje ako **koordinátor na recepcii**. Pri vysokoúrovňovej požiadavke na cestovanie
vytvára štruktúrovaný `TravelPlan` — rozkladá požiadavku na podúlohy, nastavuje priority
a identifikuje závislosti, aby concierge alebo výkonná vrstva mohli vykonať prácu.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Vykonávanie plánu so špecializovanými nástrojmi

Akonáhle recepčný agent vytvorí štruktúrovaný plán, **koncierž agent** ho vykonáva.
Každý špecializovaný nástroj spracováva jednu kategóriu podúloh (lety, hotely, aktivity). Koncierž
prechádza podúlohy plánu v poradí podľa závislostí a odosiela každú z nich do
príslušného nástroja.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Zhrnutie

V tejto lekcii ste sa naučili **plánovací návrhový vzor** pre AI agentov:

1. **Dezaggregácia úloh** — Agent na recepcii rozbije zložitú žiadosť o cestovanie na
   štruktúrované podúlohy pomocou modelov Pydantic, pričom každú priradí špecializovanému agentovi s prioritami
   a závislosťami.
2. **Štruktúrovaný výstup** — Odovzdaním `response_format` agent vracia overený
   objekt `TravelPlan` namiesto voľného textu, čo zaisťuje spoľahlivé spracovanie v ďalších krokoch.
3. **Realizácia plánu** — Concierge agent prechádza podúlohami pomocou špecializovaných nástrojov
   (`book_flight`, `reserve_hotel`, `book_activity`) na vykonanie plánu a hlásenie výsledkov.

Tento vzor oddeľuje *čo robiť* (plánovanie) od *ako to robiť* (realizácia), čím robí agentov
modulárnejšími, testovateľnými a jednoduchšie rozšíriteľnými.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
